In [ ]:
import torch
import json
import re
import ast
import logging
import zlib
import base64
import uuid
from pathlib import Path
from typing import Tuple, List, Dict
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from dataclasses import dataclass, field
from enum import Enum
from IPython.display import Image, display

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ==========================================
# 1. ENUMS AND DATA CLASSES
# ==========================================
class DiagramType(str, Enum):
    C4_CONTEXT = "c4_context"
    C4_CONTAINER = "c4_container"
    ERD = "erd"
    SEQUENCE = "sequence"
    BPMN = "bpmn"
    STATE = "state"
    GANTT = "gantt"
    ISHIKAWA = "ishikawa" # Fishbone
    CLASS = "class"

class NodeType(str, Enum):
    PERSON = "person"
    SYSTEM = "system"
    CONTAINER = "container"
    DATABASE = "database"
    MESSAGE_BUS = "message_bus"
    ENTITY = "entity"
    STATE = "state"
    TASK = "task"

class EdgeType(str, Enum):
    USES = "uses"

@dataclass
class Node:
    id: str
    name: str
    type: NodeType
    description: str = ""

@dataclass
class Edge:
    source: str
    target: str
    type: EdgeType
    label: str = ""

@dataclass
class UnifiedGraphISR:
    metadata: Dict[str, str] = field(default_factory=dict)
    nodes: List[Node] = field(default_factory=list)
    edges: List[Edge] = field(default_factory=list)

# ==========================================
# 2. UNIVERSAL RENDERER (Fixed MindMap)
# ==========================================
class UniversalDiagramRenderer:
    def __init__(self, output_dir: str = "./generated_diagrams"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)

    def render(self, isr: UnifiedGraphISR, output_filename: str = None) -> Dict:
        if output_filename is None:
            output_filename = f"diagram_{uuid.uuid4().hex[:8]}"

        dtype = isr.metadata.get('diagram_type', 'c4_container')

        # Switch Renderer
        if dtype == DiagramType.SEQUENCE:
            puml_content = self._generate_sequence(isr)
        elif dtype == DiagramType.ERD:
            puml_content = self._generate_erd(isr)
        elif dtype == DiagramType.STATE:
            puml_content = self._generate_state(isr)
        elif dtype == DiagramType.BPMN:
            puml_content = self._generate_activity(isr)
        elif dtype == DiagramType.ISHIKAWA:
            puml_content = self._generate_mindmap(isr)
        else:
            puml_content = self._generate_c4(isr)

        puml_path = self.output_dir / f"{output_filename}.puml"
        with open(puml_path, 'w') as f:
            f.write(puml_content)

        print(f"[RENDER] Type: {dtype.upper()} | Saved: {puml_path}")
        self.display_diagram(puml_content)
        return {"puml_path": str(puml_path), "success": True}

    def display_diagram(self, puml_content: str):
        try:
            zlibbed_str = zlib.compress(puml_content.encode('utf-8'))
            compressed_string = zlibbed_str[2:-4]
            encoded_string = base64.b64encode(compressed_string).decode('utf-8')
            mapping = str.maketrans('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/',
                                    '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-_')
            encoded_url_string = encoded_string.translate(mapping)
            url = f"https://www.plantuml.com/plantuml/png/{encoded_url_string}"
            display(Image(url=url))
        except Exception as e:
            print(f"[RENDER] Could not display image: {e}")

    # --- GENERATORS ---
    def _generate_mindmap(self, isr: UnifiedGraphISR) -> str:
        # Improved Fishbone Logic: Find the "Root Effect"
        lines = ["@startmindmap", "title Cause & Effect (Fishbone)"]

        # Heuristic: The Root is usually the 'Target' of most edges (Causes -> Effect)
        # If no edges, take the first node.
        if isr.edges:
            targets = [e.target for e in isr.edges]
            root_id = Counter(targets).most_common(1)[0][0]
            root = next((n for n in isr.nodes if n.id == root_id), isr.nodes[0])
        elif isr.nodes:
            # If extracting just nodes (no arrows), assume the first one mentioned is the effect
            root = isr.nodes[0]
        else:
            return "@startmindmap\n* No Data\n@endmindmap"

        lines.append(f"* {root.name}") # The Head of the fish

        # Add branches
        for node in isr.nodes:
            if node.id != root.id:
                lines.append(f"** {node.name}")

        lines.append("@endmindmap")
        return "\n".join(lines)

    def _generate_sequence(self, isr: UnifiedGraphISR) -> str:
        lines = ["@startuml", "autonumber", "title Sequence Diagram"]
        for node in isr.nodes: lines.append(f'participant "{node.name}" as {node.id}')
        for edge in isr.edges: lines.append(f'{edge.source} -> {edge.target} : {edge.label}')
        lines.append("@enduml")
        return "\n".join(lines)

    def _generate_erd(self, isr: UnifiedGraphISR) -> str:
        lines = ["@startuml", "hide circle", "skinparam linetype ortho", "title ER Diagram"]
        for node in isr.nodes: lines.append(f'entity "{node.name}" as {node.id} {{}}')
        for edge in isr.edges: lines.append(f'{edge.source} ||..|| {edge.target} : {edge.label}')
        lines.append("@enduml")
        return "\n".join(lines)

    def _generate_state(self, isr: UnifiedGraphISR) -> str:
        lines = ["@startuml", "title State Machine"]
        lines.append("[*] --> " + (isr.nodes[0].id if isr.nodes else "Start"))
        for node in isr.nodes: lines.append(f'state "{node.name}" as {node.id}')
        for edge in isr.edges: lines.append(f'{edge.source} --> {edge.target} : {edge.label}')
        lines.append("@enduml")
        return "\n".join(lines)

    def _generate_activity(self, isr: UnifiedGraphISR) -> str:
        lines = ["@startuml", "start", "title Process Flow"]
        for edge in isr.edges:
            lines.append(f':{edge.source};')
            lines.append(f'-> {edge.label};')
        if isr.edges: lines.append(f':{isr.edges[-1].target};')
        lines.append("stop")
        lines.append("@enduml")
        return "\n".join(lines)

    def _generate_c4(self, isr: UnifiedGraphISR) -> str:
        lines = ["@startuml", "!include https://raw.githubusercontent.com/plantuml-stdlib/C4-PlantUML/master/C4_Container.puml", "LAYOUT_WITH_LEGEND()"]
        title = isr.metadata.get('cleaned_input', 'System').replace('\n', ' ')[:60]
        lines.append(f"title {title}")
        type_map = {NodeType.PERSON: "Person", NodeType.DATABASE: "ContainerDb", NodeType.MESSAGE_BUS: "ContainerQueue", NodeType.SYSTEM: "System", NodeType.CONTAINER: "Container"}
        for node in isr.nodes:
            c4 = type_map.get(node.type, "Container")
            lines.append(f'{c4}({node.id}, "{node.name}", "Tech")')
        for edge in isr.edges: lines.append(f'Rel({edge.source}, {edge.target}, "{edge.label}")')
        lines.append("@enduml")
        return "\n".join(lines)

# ==========================================
# 3. PIPELINE (Fixed Classifier Order)
# ==========================================
class DiagramGenerationPipeline:
    def __init__(self, model_name: str = "google/flan-t5-large", device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[INIT] Loading model: {model_name} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
        self.renderer = None

    def generate_diagram(self, user_input: str):
        logger.info(f"[PIPELINE] Processing...")

        # 1. Classify (Improved)
        diagram_type = self._classify_intent(user_input)
        logger.info(f"[INTENT] Detected: {diagram_type.value.upper()}")

        # 2. Extract
        extracted = self._extract_structure(user_input)
        rels = extracted.get('relationships', [])

        # 3. Build Nodes/Edges
        nodes = self._create_nodes(rels, user_input)
        if not nodes: nodes = self._manual_fallback(user_input)

        edges = self._create_edges(rels, nodes, user_input)
        if not edges: edges = self._regex_edge_fallback(user_input, nodes)

        # 4. Render
        isr = UnifiedGraphISR(
            metadata={"user_input": user_input, "cleaned_input": user_input, "diagram_type": diagram_type},
            nodes=nodes, edges=edges
        )
        if self.renderer: self.renderer.render(isr)
        return isr

    def _classify_intent(self, text: str) -> DiagramType:
        t = text.lower()
        # Specific types MUST be checked BEFORE generic types like 'database'
        if "fishbone" in t or "ishikawa" in t or "cause" in t and "effect" in t: return DiagramType.ISHIKAWA
        if "gantt" in t or "timeline" in t: return DiagramType.GANTT
        if "bpmn" in t or "swimlane" in t: return DiagramType.BPMN
        if "state" in t and "machine" in t: return DiagramType.STATE
        if "sequence" in t: return DiagramType.SEQUENCE

        # Generic types
        if "database" in t or "erd" in t or "entity" in t: return DiagramType.ERD
        return DiagramType.C4_CONTAINER

    def _extract_structure(self, text: str) -> Dict:
        prompt = f"""Identify connections. Format: A -> B. Input: "{text}" Connections:"""
        try:
            inputs = self.tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(self.device)
            outputs = self.model.generate(**inputs, max_length=256, num_beams=4)
            result = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            rels = []
            parts = re.split(r',|\sand\s|\.', result)
            verb_pattern = r'(.*?)\s+(?:to|->|uses|calls|connects|publishes|triggers|leads to|contributes to|causes)\s+(.*)'
            for part in parts:
                match = re.search(verb_pattern, part, re.IGNORECASE)
                if match:
                    src, dst = match.group(1).strip(), match.group(2).strip()
                    if len(src)<40 and len(dst)<40: rels.append({"from": src, "to": dst})
            return {"relationships": rels}
        except: return {"relationships": []}

    def _create_nodes(self, rels: List[Dict], text: str) -> List[Node]:
        names = set()
        for r in rels: names.add(r['from']); names.add(r['to'])
        nodes = []
        for name in names:
            nid = name.lower().replace(' ', '_').replace('-', '_')
            nodes.append(Node(id=nid, name=name, type=NodeType.SYSTEM))
        return nodes

    def _create_edges(self, rels: List[Dict], nodes: List[Node], text: str) -> List[Edge]:
        edges = []
        node_map = {n.name: n.id for n in nodes}
        for n in nodes: node_map[n.id] = n.id
        for rel in rels:
            src = next((node_map[k] for k in node_map if k in rel['from']), None)
            dst = next((node_map[k] for k in node_map if k in rel['to']), None)
            if src and dst: edges.append(Edge(source=src, target=dst, type=EdgeType.USES, label="causes"))
        return edges

    def _manual_fallback(self, text: str) -> List[Node]:
        words = re.findall(r'\b[A-Za-z0-9]+\b', text)
        blacklist = ["The", "A", "An", "Is", "To", "From", "And", "Design", "Generate", "Diagram", "For", "Include", "Another"]
        potential = []
        i = 0
        while i < len(words):
            w = words[i]
            if w[0].isupper() and w not in blacklist and len(w)>1:
                if i+1 < len(words) and words[i+1][0].isupper():
                    potential.append(f"{w} {words[i+1]}")
                    i += 2; continue
                potential.append(w)
            i += 1
        return [Node(id=p.lower().replace(' ', '_'), name=p, type=NodeType.SYSTEM) for p in set(potential)]

    def _regex_edge_fallback(self, text: str, nodes: List[Node]) -> List[Edge]:
        # For Fishbone: Everything connects to the 'Effect' (last node or central node)
        edges = []
        if len(nodes) > 1:
            # Assume the first one is the Effect (Head) and others are causes
            # Or assume the last one.
            # Let's try to find the word "Effect" or "Result" in the node names
            effect_node = next((n for n in nodes if "effect" in n.id or "result" in n.id or "slowdown" in n.id), nodes[0])
            for n in nodes:
                if n.id != effect_node.id:
                    edges.append(Edge(source=n.id, target=effect_node.id, type=EdgeType.USES, label="causes"))
        return edges

# ==========================================
# 4. EXECUTION
# ==========================================
print("✅ Universal Renderer Updated.")
pipeline = DiagramGenerationPipeline()
pipeline.renderer = UniversalDiagramRenderer()

# RETRY FISHBONE
user_input_fishbone = """
Generate a Fishbone diagram for the effect: Website Slowdown.
Causes include High CPU Usage.
Another cause is Memory Leak.
Network Latency leads to the slowdown.
Database Locks also contribute to the effect.
"""
pipeline.generate_diagram(user_input_fishbone)

✅ Universal Renderer Updated.
[INIT] Loading model: google/flan-t5-large on cpu...


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


[RENDER] Type: ISHIKAWA | Saved: generated_diagrams/diagram_595b2d14.puml


UnifiedGraphISR(metadata={'user_input': '\nGenerate a Fishbone diagram for the effect: Website Slowdown.\nCauses include High CPU Usage.\nAnother cause is Memory Leak.\nNetwork Latency leads to the slowdown.\nDatabase Locks also contribute to the effect.\n', 'cleaned_input': '\nGenerate a Fishbone diagram for the effect: Website Slowdown.\nCauses include High CPU Usage.\nAnother cause is Memory Leak.\nNetwork Latency leads to the slowdown.\nDatabase Locks also contribute to the effect.\n', 'diagram_type': <DiagramType.ISHIKAWA: 'ishikawa'>}, nodes=[Node(id='memory_leak', name='Memory Leak', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='usage_another', name='Usage Another', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='causes', name='Causes', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='fishbone', name='Fishbone', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='network_latency', name='Network Latency', type=<NodeType.SYSTEM: 'system

In [ ]:
user_input = """
Show me a sequence diagram where:
The User calls the Login API.
The Login API queries the User Database.
The User Database returns the User Profile.
The Login API sends a Token to the User.
"""
pipeline.generate_diagram(user_input)

[RENDER] Type: SEQUENCE | Saved: generated_diagrams/diagram_316e8fac.puml


UnifiedGraphISR(metadata={'user_input': '\nShow me a sequence diagram where:\nThe User calls the Login API.\nThe Login API queries the User Database.\nThe User Database returns the User Profile.\nThe Login API sends a Token to the User.\n', 'cleaned_input': '\nShow me a sequence diagram where:\nThe User calls the Login API.\nThe Login API queries the User Database.\nThe User Database returns the User Profile.\nThe Login API sends a Token to the User.\n', 'diagram_type': <DiagramType.SEQUENCE: 'sequence'>}, nodes=[Node(id='the_user', name='the User', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='the_login_api', name='the Login API', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='the_login_api_sends_a_token', name='The Login API sends a Token', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='the_user', name='The User', type=<NodeType.SYSTEM: 'system'>, description='')], edges=[Edge(source='the_user', target='the_login_api', type=<EdgeType.USES: 'use

In [ ]:
user_input_erd = """
Design a Database Schema for a University System.
The Student table has a relation to the Course table.
The Course table connects to the Professor table.
The Student table also references the Grade table.
"""
pipeline.generate_diagram(user_input_erd)

[RENDER] Type: ERD | Saved: generated_diagrams/diagram_944212ba.puml


UnifiedGraphISR(metadata={'user_input': '\nDesign a Database Schema for a University System.\nThe Student table has a relation to the Course table.\nThe Course table connects to the Professor table.\nThe Student table also references the Grade table.\n', 'cleaned_input': '\nDesign a Database Schema for a University System.\nThe Student table has a relation to the Course table.\nThe Course table connects to the Professor table.\nThe Student table also references the Grade table.\n', 'diagram_type': <DiagramType.ERD: 'erd'>}, nodes=[Node(id='to_the_professor_table', name='to the Professor table', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='the_course_table', name='The Course table', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='the_student_table_has_a_relation', name='The Student table has a relation', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='the_course_table', name='the Course table', type=<NodeType.SYSTEM: 'system'>, description='')], ed

In [ ]:
input_state_atm = """
Show me the State Machine for an ATM Session.
The ATM starts in the Idle state.
When the User inserts a card, it transitions to Authenticating.
If the PIN is correct, it transitions to TransactionMenu.
If the User selects Withdraw, it transitions to DispensingCash.
After DispensingCash, it returns to EjectCard.
From EjectCard, it returns to Idle.
"""
pipeline.generate_diagram(input_state_atm)

[RENDER] Type: STATE | Saved: generated_diagrams/diagram_72dcaa0f.puml


UnifiedGraphISR(metadata={'user_input': '\nShow me the State Machine for an ATM Session.\nThe ATM starts in the Idle state.\nWhen the User inserts a card, it transitions to Authenticating.\nIf the PIN is correct, it transitions to TransactionMenu.\nIf the User selects Withdraw, it transitions to DispensingCash.\nAfter DispensingCash, it returns to EjectCard.\nFrom EjectCard, it returns to Idle.\n', 'cleaned_input': '\nShow me the State Machine for an ATM Session.\nThe ATM starts in the Idle state.\nWhen the User inserts a card, it transitions to Authenticating.\nIf the PIN is correct, it transitions to TransactionMenu.\nIf the User selects Withdraw, it transitions to DispensingCash.\nAfter DispensingCash, it returns to EjectCard.\nFrom EjectCard, it returns to Idle.\n', 'diagram_type': <DiagramType.STATE: 'state'>}, nodes=[Node(id='dispensingcash', name='DispensingCash', type=<NodeType.SYSTEM: 'system'>, description=''), Node(id='transactionmenu', name='TransactionMenu', type=<NodeType